# Diabetes Prediction with PyTorch From Scratch

This notebook walks through a beginner-friendly binary classification project using a real diabetes dataset and a small PyTorch neural network.

## What you will do

In this lesson, you will move through the full machine learning workflow step by step:
- load a CSV dataset
- inspect the data
- convert text columns into numeric values
- standardize continuous features
- convert the data into PyTorch tensors
- build and train a neural network
- make a sample prediction

## What you should focus on

Try to understand what each cell is doing before moving to the next one. The goal is not just to run the notebook, but to understand why each preparation step matters.

## Prerequisites

You do not need advanced math for this notebook. It helps if you already know:
- basic Python syntax
- how to run notebook cells
- how tables look in pandas

Each section includes short explanations so the notebook feels more like a guided lesson than a script.

## 0. Point to the dataset

Before loading anything, we store the dataset path in a variable. This makes the rest of the notebook easier to read because every later cell can reuse the same path.

This cell does not load the data yet. It only tells Python where the CSV file lives.

In [ ]:
# Store the dataset path once so the rest of the notebook can reuse it.
dataset_path = r"C:\projects\pytorch\project\dataset\diabetes_prediction_dataset.csv"

# Print the path to confirm that we are pointing to the expected file.
print("Dataset path:", dataset_path)

Dataset path: C:\projects\pytorch\project\dataset\diabetes_prediction_dataset.csv


## 1. Import the libraries and set random seeds

PyTorch is the main library for building the neural network, pandas helps us work with the CSV file, and NumPy helps with numeric operations.

We also set random seeds so the notebook gives similar results each time you rerun it. That makes learning and debugging much easier.

In [ ]:
# Import the core libraries used throughout this notebook.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Set random seeds so results stay as consistent as possible between runs.
torch.manual_seed(42)
np.random.seed(42)

## 2. Load the CSV and preview the raw data

Now we read the dataset into a pandas DataFrame. After that, we display the first few rows so you can see the columns, the value types, and the general shape of the data before any cleaning.

In [ ]:
# Load the CSV file into a pandas DataFrame.
df = pd.read_csv(dataset_path)

# Show the first 10 rows so we can inspect the raw dataset.
df.head(10)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
5,Female,20.0,0,0,never,27.32,6.6,85,0
6,Female,44.0,0,0,never,19.31,6.5,200,1
7,Female,79.0,0,0,No Info,23.86,5.7,85,0
8,Male,42.0,0,0,never,33.64,4.8,145,0
9,Female,32.0,0,0,never,27.32,5.0,100,0


## 3. Turn text categories into numeric values

Neural networks work with numbers, not text labels. In this dataset, columns like gender and smoking history are stored as words, so we convert them into integer codes before training a model.

In [ ]:
# Convert the text-based columns into integer codes that PyTorch can work with.
encoded_df = df.replace({
    "gender": {"Female": 0, "Male": 1, "Other": 2},
    "smoking_history": {
        "No Info": 0,
        "never": 1,
        "former": 2,
        "current": 3,
        "not current": 4,
        "ever": 5,
    },
})

# Make sure the encoded columns are stored as integers, not objects.
encoded_df[["gender", "smoking_history"]] = encoded_df[["gender", "smoking_history"]].astype("int64")

# Preview the first 10 rows after encoding.
encoded_df.head(10)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,0,80.0,0,1,1,25.19,6.6,140,0
1,0,54.0,0,0,0,27.32,6.6,80,0
2,1,28.0,0,0,1,27.32,5.7,158,0
3,0,36.0,0,0,3,23.45,5.0,155,0
4,1,76.0,1,1,3,20.14,4.8,155,0
5,0,20.0,0,0,1,27.32,6.6,85,0
6,0,44.0,0,0,1,19.31,6.5,200,1
7,0,79.0,0,0,0,23.86,5.7,85,0
8,1,42.0,0,0,1,33.64,4.8,145,0
9,0,32.0,0,0,1,27.32,5.0,100,0


## 4. Inspect the encoded table

It is useful to pause and look at the full prepared DataFrame after encoding. This makes it easier to confirm that the categorical columns are now numeric and that the target column is still present.

In [ ]:
# Display the full encoded DataFrame so we can inspect all columns together.
encoded_df

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,0,80.0,0,1,1,25.19,6.6,140,0
1,0,54.0,0,0,0,27.32,6.6,80,0
2,1,28.0,0,0,1,27.32,5.7,158,0
3,0,36.0,0,0,3,23.45,5.0,155,0
4,1,76.0,1,1,3,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
99995,0,80.0,0,0,0,27.32,6.2,90,0
99996,0,2.0,0,0,0,17.37,6.5,100,0
99997,1,66.0,0,0,2,27.83,5.7,155,0
99998,0,24.0,0,0,1,35.42,4.0,100,0


## 5. Standardize the continuous features

Feature scaling helps neural networks train more smoothly. Here we use standardization for continuous columns such as age, BMI, HbA1c level, and blood glucose level.

### Quick intuition

Normalization is often used when values live in a fixed range, such as image pixels from 0 to 255.

Standardization is a better fit here because these medical measurements have different ranges and units. It rescales each column using:

$(value - mean) / standard\ deviation$

In [ ]:
# Separate the input features from the target column.
features = encoded_df.drop("diabetes", axis=1).copy()
labels = encoded_df["diabetes"].copy()

# These columns are continuous and should be standardized.
continuous_columns = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
binary_or_encoded_columns = ["gender", "hypertension", "heart_disease", "smoking_history"]

# Look at the current scale of the continuous columns before standardization.
display(features[continuous_columns].describe().T[["mean", "std", "min", "max"]])

# Compute the mean and standard deviation for each continuous column.
feature_means = features[continuous_columns].mean()
feature_stds = features[continuous_columns].std()

# Create a copy and standardize only the continuous columns.
standardized_features = features.copy()
standardized_features[continuous_columns] = (
    standardized_features[continuous_columns] - feature_means
) / feature_stds

# Preview the transformed features.
standardized_features.head(10)

,mean,std,min,max
age,41.885856,22.516840,0.08,80.00
bmi,27.320767,6.636783,10.01,95.69
HbA1c_level,5.527507,1.070672,3.50,9.00
blood_glucose_level,138.058060,40.708136,80.00,300.00


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level
0,0,1.692695,0,1,1,-0.321054,1.001701,0.047704
1,0,0.538004,0,0,0,-0.000116,1.001701,-1.426203
2,1,-0.616688,0,0,1,-0.000116,0.161107,0.489876
3,0,-0.261398,0,0,3,-0.583229,-0.492688,0.416181
4,1,1.515050,1,1,3,-1.081965,-0.679486,0.416181
5,0,-0.971977,0,0,1,-0.000116,1.001701,-1.303377
6,0,0.093892,0,0,1,-1.207026,0.908301,1.521611
7,0,1.648284,0,0,0,-0.521452,0.161107,-1.303377
8,1,0.005069,0,0,1,0.952153,-0.679486,0.170530
9,0,-0.439043,0,0,1,-0.000116,-0.492688,-0.934901


## 6. Convert the prepared data into PyTorch tensors

PyTorch models expect tensors instead of pandas objects. In this step, we convert the feature table and the target column into tensor form and check their shapes so we know they are ready for training.

In [ ]:
# Convert the standardized feature table into a float32 tensor.
features_tensor = torch.tensor(
    standardized_features.to_numpy(dtype=np.float32),
    dtype=torch.float32,
)

# Convert the labels into a float32 tensor with one column.
labels_tensor = torch.tensor(
    labels.to_numpy(dtype=np.float32).reshape(-1, 1),
    dtype=torch.float32,
)

# Print shapes and data types to confirm the tensors are ready for training.
print("Features tensor shape:", features_tensor.shape)
print("Features tensor dtype:", features_tensor.dtype)
print("Labels tensor shape:", labels_tensor.shape)
print("Labels tensor dtype:", labels_tensor.dtype)

Features tensor shape: torch.Size([100000, 8])
Features tensor dtype: torch.float32
Labels tensor shape: torch.Size([100000, 1])
Labels tensor dtype: torch.float32


## 7. Build and train a simple neural network

This model is a small feed-forward neural network for binary classification. It takes the prepared features as input and learns to predict whether a patient belongs to the diabetes class.

The training loop repeats the same pattern many times: forward pass, loss calculation, backward pass, parameter update, and gradient reset.


In [ ]:
# Build a simple feed-forward neural network for binary classification.
model = torch.nn.Sequential(
    torch.nn.Linear(8, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 1),
)

# Use binary cross-entropy with logits because this is a yes-or-no prediction task.
loss = torch.nn.BCEWithLogitsLoss()
LR = 0.01

# Train for a fixed number of iterations.
n = 100
for i in range(n):
    # Forward pass: compute predictions from the current model parameters.
    y_pred = model(features_tensor)

    # Measure how far the predictions are from the true labels.
    l = loss(y_pred, labels_tensor)

    # Backward pass: compute gradients for every trainable parameter.
    l.backward()

    # Update each parameter using gradient descent.
    # This is actually the work of the optimizer to update the weights, but we're doing it manually here to see how it works under the hood.
    with torch.no_grad():
        for param in model.parameters():
            param -= LR * param.grad

    # Clear old gradients before the next training step.
    ##### optimizer.zero_grad() + optimizer.step() is the normal production PyTorch workflow.
    ##### Manual param update + model.zero_grad() is useful for understanding what the optimizer is doing under the hood.   
    model.zero_grad()

# Show the final training loss so we have a quick sense of training progress.
print("Final training loss:", l.item())

## 8. Make a sample prediction

After training, it is useful to test the model on a single row. This helps connect the training step to a concrete prediction: the model produces a logit, we convert it into a probability, and then we apply a threshold to choose a class label.

In [ ]:
# Choose the probability cutoff used to turn probabilities into class labels.
threshold = 0.5

# Select one example row and add a batch dimension because the model expects batches.
sample_input = features_tensor[1].unsqueeze(0)
print("Input shape:", sample_input.shape)

with torch.no_grad():
    # Run the sample through the model to get a raw prediction score.
    logits = model(sample_input)

    # Convert the raw score into a probability between 0 and 1.
    probability = torch.sigmoid(logits).item()

    # Turn the probability into a class prediction using the threshold.
    predicted_class = int(probability >= threshold)

print("Predicted probability:", probability)
print("Threshold:", threshold)
print("Predicted class:", predicted_class)

Input shape: torch.Size([1, 8])
Predicted probability: 0.28328368067741394
Threshold: 0.5
Predicted class: 0
